# Experimento completo (Colab, GPU T4)

Corre el experimento de punta a punta, deja las trazas en `resultados/` y genera las tablas, la figura y las cifras del informe (`notebook.ipynb`).

| Paso | Qué hace | Tiempo aprox. en T4 |
|---|---|---|
| 0 | Preparar: GPU, repo, vLLM | 3–5 min |
| 1 | Prueba de formato (5 preguntas, 1.5B) | 3 min |
| 2 | Dev (20 preguntas, 1.5B y 7B) | 6 min |
| 3 | **Test** (160 preguntas, 1.5B, 3B y 7B): resultados del informe | 35–45 min |
| 4 | Extra: 14B (solo directa y react) | 30–35 min |
| 5 | Tablas, figura y cifras del informe (sin GPU) | 1 min |
| 6 | Guardar resultados | 1 min |

Cada `correr.py` corre en un proceso aparte, porque vLLM no libera bien la GPU dentro del notebook. Cada corrida escribe `resultados/trazas_<modelo>_<split>.jsonl` y reemplaza solo las condiciones que vuelve a correr. Los pasos 1 y 2 son opcionales: los prompts se congelaron con dev, antes de correr test.

## 0. Preparar

Requiere GPU: *Entorno de ejecución → Cambiar tipo de entorno → T4*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Clona el repo (o lo actualiza si ya estaba clonado en esta sesión).
import os
if not os.path.exists('/content/proyecto'):
    !git clone -q https://github.com/Felipedino/ReAct-Rag-Multimodal.git /content/proyecto
os.chdir('/content/proyecto')
!git pull -q
!git log --oneline -1

In [ ]:
# Versión probada: vllm 0.34.0 (trae torch 2.14.0+cu130).
!pip install -q vllm==0.34.0
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__)"

## 1. Prueba de formato (opcional)

5 preguntas escritas a mano (`data/prueba.jsonl`). Sirve para revisar a ojo el prompt y el parseo de las acciones. **No** mide desempeño.

Primero se arma el prompt de ReAct usando solo el tokenizer, sin cargar el modelo en la GPU:

In [ ]:
import json
from transformers import AutoTokenizer
import llm, agents

class SoloTokenizer:
    # Imita a vllm.LLM lo justo para armar prompts sin usar la GPU.
    def __init__(self, nombre):
        self.tok = AutoTokenizer.from_pretrained(nombre)
    def get_tokenizer(self):
        return self.tok

falso = SoloTokenizer(llm.MODELOS['1.5B'])
with open('data/prueba.jsonl', encoding='utf-8') as f:
    q = json.loads(f.readline())
usuario = f"{agents.INSTR_REACT}\nEjemplos:\n{agents.EJEMPLOS_REACT}\nPregunta: {q['pregunta']}"
p = llm.armar_prompt(falso, agents.SISTEMA, usuario, prefijo_asistente='Pensamiento 1:')
print(p)
print('Tokens del prompt ReAct (paso 1):', llm.contar_tokens(falso, p))

In [ ]:
!python correr.py 1.5B prueba
!python evaluate.py prueba

In [ ]:
# Trazas de la prueba: respuesta, cómo terminó y, en ReAct, la trayectoria completa.
with open('resultados/trazas_1.5B_prueba.jsonl', encoding='utf-8') as f:
    trazas = [json.loads(l) for l in f]
for t in trazas:
    print('=' * 100)
    print(f"[{t['condicion']}] {t['pregunta']}")
    print(f"gold: {t['respuesta_gold']} | modelo: {t['respuesta_modelo']!r} | "
          f"termino: {t['termino']} | acciones inválidas: {t['acciones_invalidas']}")
    if t['condicion'] == 'react':
        print(t['texto_modelo'])

Qué mirar:
- directa y rag: respuestas cortas, sin explicación.
- react: cada paso trae `Acción k: Buscar[...]` o `Terminar[...]`, y el modelo no inventa observaciones.
- `termino='contexto'` indica que hay que subir `llm.MAX_MODEL_LEN`.

## 2. Dev (opcional)

20 preguntas (5 por celda). Sirvió para revisar que todo funcione y congelar los prompts antes del test.

In [ ]:
!python correr.py 1.5B dev
!python correr.py 7B dev
!python evaluate.py dev

In [ ]:
# ReAct del 1.5B que llegó al máximo de pasos (bucles): qué buscó en cada paso.
with open('resultados/trazas_1.5B_dev.jsonl', encoding='utf-8') as f:
    tr = [json.loads(l) for l in f]
bucles = [t for t in tr if t['condicion'] == 'react' and t['termino'] == 'max_pasos']
print(f'{len(bucles)} de {sum(t["condicion"] == "react" for t in tr)} trazas de ReAct llegan a 7 pasos')
for t in bucles[:3]:
    print('=' * 80); print(t['pregunta'], '| gold:', t['respuesta_gold'])
    for i, paso in enumerate(t['pasos'], 1):
        print(f"{i}. {paso['accion']}[{paso['argumento']}] -> {paso['titulo_observado']}")

## 3. Test (resultados del informe)

160 preguntas (40 por celda) y 3 condiciones (directa, rag, react). Un modelo por celda: si la sesión se corta, se retoma desde el modelo que falta. Al final de cada corrida, `correr.py` imprime el tiempo, los tokens generados y las llamadas reales a Wikipedia.

In [ ]:
!python correr.py 1.5B test     # ~10-12 min

In [ ]:
!python correr.py 3B test       # ~10 min

In [ ]:
!python correr.py 7B test       # ~11-13 min (incluye descarga)

In [ ]:
!python evaluate.py test

## 4. Extra: Qwen2.5-14B-Instruct-AWQ (opcional)

No es parte del diseño original (1.5B, 3B y 7B); sirve para ver si la tendencia con el tamaño sigue. Tiene tres diferencias con el resto:

- **Solo directa y react.** RAG fijo se omite por tiempo; ReAct solo toma ~26 min.
- **`MAX_MODEL_LEN = 6144`** en vez de 8192. El 14B pesa ~9,6 GB y deja ~7.000 tokens de caché KV (con `gpu_memory_utilization=0.95`, ya fijado en `llm.py`). vLLM exige que quepa al menos una petición del largo máximo. Consecuencia: ReAct puede cortarse por contexto antes que en los otros modelos (ver `termino == 'contexto'` en las trazas).
- El cambio de largo se hace con `sed` sobre `llm.py` solo para esta corrida, y después se restaura con `git checkout`.

Tiempo: ~4 min de carga + ~26 min de generación.

In [ ]:
!sed -i 's/^MAX_MODEL_LEN = 8192/MAX_MODEL_LEN = 6144/' llm.py
!grep -n "^MAX_MODEL_LEN" llm.py
!python correr.py 14B test directa react
!git checkout llm.py              # vuelve a 8192 para el resto
!python evaluate.py test

In [ ]:
# ¿Cuántas trazas de ReAct del 14B se cortaron por contexto?
import collections
with open('resultados/trazas_14B_test.jsonl', encoding='utf-8') as f:
    t14 = [json.loads(l) for l in f]
print(collections.Counter(t['termino'] for t in t14 if t['condicion'] == 'react'))

## 5. Tablas, figura y cifras del informe

Se calcula todo desde las trazas de `resultados/` (no usa GPU). Si se saltaron los pasos 3–4, usa las trazas guardadas en el repositorio. Primero `evaluate.py test` escribe `tabla_test.csv`, `ganancia_test.csv` y `errores_test.csv`.

In [ ]:
!python evaluate.py test > /dev/null
import collections, json, re
import pandas as pd
from IPython.display import display, Image
import evaluate

tab = pd.read_csv('resultados/tabla_test.csv')
gan = pd.read_csv('resultados/ganancia_test.csv')
MODELOS = [m for m in evaluate.ORDEN_MODELOS if m in set(tab['modelo'])]

def con_ic(df, col, lo, hi):
    return [f'{v:.2f} [{a:.2f}–{b:.2f}]' for v, a, b in zip(df[col], df[lo], df[hi])]

total = tab[tab['agrupacion'] == 'total'].copy()
celdas = tab[tab['agrupacion'] == 'celda']
total['EM'] = con_ic(total, 'em', 'em_lo', 'em_hi')
print('EM total [IC 95 %]:')
display(total.pivot(index='modelo', columns='condicion', values='EM')
        .reindex(index=MODELOS, columns=evaluate.ORDEN_CONDICIONES))
print('F1 total:')
display(total.pivot(index='modelo', columns='condicion', values='f1')
        .reindex(index=MODELOS, columns=evaluate.ORDEN_CONDICIONES).round(2))
g = gan[(gan['comparacion'] == 'react - directa') & (gan['agrupacion'] != 'celda')].copy()
g['ganancia'] = con_ic(g, 'ganancia_em', 'ic_lo', 'ic_hi')
print('Ganancia ReAct − Directa (EM, bootstrap pareado):')
display(g.pivot(index='modelo', columns='grupo', values='ganancia')
        .reindex(index=MODELOS, columns=['total', 'chile', 'no_chile', 'pocos', 'muchos']))
print('EM por celda:')
display(celdas.pivot_table(index=['modelo', 'condicion'], columns='grupo', values='em')
        .reindex(columns=evaluate.ORDEN_CELDAS).round(3))

Figura principal: (a) EM total por tamaño y condición; (b) EM de ReAct por celda, con una raya negra para Directa del modelo más grande.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ETIQUETA = {m: m.replace('.', ',') for m in MODELOS}
NOMBRE = {'directa': 'Directa (memoria)', 'rag': 'RAG fijo (1 búsqueda)', 'react': 'ReAct'}
COLOR = {'directa': '#888888', 'rag': '#4a76a8', 'react': '#e45756'}
ROJOS = (['#f4a6a6', '#e45756', '#9b1c1c'] if len(MODELOS) <= 3
         else ['#f8c9c9', '#ee8f8f', '#d9473f', '#9b1c1c'])   # más oscuro = más grande

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14.5, 5.2))
for cond in evaluate.ORDEN_CONDICIONES:
    f = total[total['condicion'] == cond].set_index('modelo').reindex(MODELOS)
    ax1.errorbar(range(len(MODELOS)), f['em'], yerr=[f['em'] - f['em_lo'], f['em_hi'] - f['em']],
                 marker='o', lw=2.2, capsize=4, color=COLOR[cond], label=NOMBRE[cond])
ax1.set_xticks(range(len(MODELOS)), [ETIQUETA[m] for m in MODELOS])
ax1.set_xlabel('Tamaño del modelo (Qwen2.5, AWQ)')
ax1.set_ylabel(f'Exact Match (test, n={int(total["n"].iloc[0])})')
ax1.set_title('(a) Solo ReAct escala con el tamaño')
ax1.legend(frameon=False)

react = celdas[celdas['condicion'] == 'react']
ancho = 0.8 / len(MODELOS)
for k, m in enumerate(MODELOS):
    f = react[react['modelo'] == m].set_index('grupo').reindex(evaluate.ORDEN_CELDAS)
    xs = [i + (k - (len(MODELOS) - 1) / 2) * ancho for i in range(4)]
    ax2.bar(xs, f['em'], width=ancho, color=ROJOS[k], label=ETIQUETA[m],
            yerr=[f['em'] - f['em_lo'], f['em_hi'] - f['em']], capsize=3)
mayor = MODELOS[-1]
d = celdas[(celdas['condicion'] == 'directa') & (celdas['modelo'] == mayor)] \
    .set_index('grupo').reindex(evaluate.ORDEN_CELDAS)
x_mayor = (len(MODELOS) - 1) / 2 * ancho
ax2.hlines(d['em'], [i + x_mayor - ancho / 3 for i in range(4)], [i + x_mayor + ancho / 3 for i in range(4)],
           color='black', lw=2, label=f'Directa {ETIQUETA[mayor]}')
ax2.set_xticks(range(4), ['Chile\npocos', 'Chile\nmuchos', 'No Chile\npocos', 'No Chile\nmuchos'])
ax2.set_ylabel(f'Exact Match de ReAct (n={int(react["n"].iloc[0])} por celda)')
ax2.set_title('(b) ReAct por celda (IC 95 % bootstrap)')
ax2.legend(frameon=False, ncol=len(MODELOS) + 1, loc='upper center', fontsize=9, handlelength=1.5)
for ax in (ax1, ax2):
    ax.grid(axis='y', alpha=0.3)
    ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax1.set_ylim(0, 1)
ax2.set_ylim(0, 1.08)   # espacio arriba para la leyenda
ax1.grid(axis='x', alpha=0.3)
fig.tight_layout()
fig.savefig('resultados/figura_principal.png', dpi=150)
plt.close(fig)
display(Image('resultados/figura_principal.png'))

Cifras de la sección de errores del informe:

- **Tipos de error de ReAct**, según la clasificación automática de `evaluate.py`.
- **Uso de las sugerencias:** una búsqueda fallida devuelve 5 títulos similares; se cuenta si el paso siguiente busca uno de ellos y si B estaba entre ellos.
- **Alucinaciones:** fallo con una respuesta concreta que no aparece en nada de lo que el modelo leyó. Se excluyen los fallos sin respuesta, las abstenciones y los aciertos parciales.

In [ ]:
err = pd.read_csv('resultados/errores_test.csv')
print('Tipos de error de ReAct:')
display(err[err['condicion'] == 'react'].groupby(['tipo', 'modelo']).size()
        .unstack(fill_value=0).reindex(columns=MODELOS))

with open('data/dataset.jsonl', encoding='utf-8') as f:
    titulo_B = {q['id']: q['titulo_B'] for q in map(json.loads, f)}
TRAZAS = {}
for m in MODELOS:
    with open(f'resultados/trazas_{m}_test.jsonl', encoding='utf-8') as f:
        TRAZAS[m] = [evaluate.puntuar(json.loads(l)) for l in f]

SIMILARES = re.compile(r"Similares: \[(.*)\]\.?\s*$", re.S)
ABSTENCION = re.compile(r"\b(no se|no encontr|no pud|informacion|desconocid|titulos|insuficiente)")

def en_lista(titulo, lista):
    # Compara elementos completos de "a, b, c" (los títulos pueden tener comas).
    return bool(titulo) and f', {titulo.strip().lower()}, ' in f', {lista.lower()}, '

def clase_de_fallo(t):
    resp = (t['respuesta_modelo'] or '').strip()
    normal = ' '.join(evaluate.tokens(resp))
    if not resp:
        return 'sin respuesta'
    if not normal or ABSTENCION.search(normal):
        return 'se abstiene'
    if t['contiene']:
        return 'acierto parcial'
    leido = evaluate.observaciones(t)
    return 'responde algo leído' if any(evaluate.contiene(o, resp) for o in leido) else 'alucinación'

filas = []
for m in MODELOS:
    react = [t for t in TRAZAS[m] if t['condicion'] == 'react']
    fallidas = usa = b_sugerido = b_buscado = 0
    for t in react:
        pasos = t['pasos']
        for i, p in enumerate(pasos):
            hallado = SIMILARES.search(p.get('observacion') or '')
            if p['accion'] != 'Buscar' or p.get('titulo_observado') or not hallado:
                continue
            fallidas += 1
            sig = pasos[i + 1] if i + 1 < len(pasos) else {}
            arg = sig.get('argumento') or ''
            busca = sig.get('accion') == 'Buscar'
            usa += busca and en_lista(arg, hallado.group(1))
            if en_lista(titulo_B[t['id']], hallado.group(1)):
                b_sugerido += 1
                b_buscado += busca and arg.strip().lower() == titulo_B[t['id']].lower()
    fallos = [t for t in react if t['em'] == 0]
    clases = collections.Counter(clase_de_fallo(t) for t in fallos)
    filas.append({'modelo': m, 'búsquedas fallidas': fallidas,
                  'usa sugerencia': f'{usa / fallidas:.0%}',
                  'B sugerido': b_sugerido, 'B buscado': b_buscado,
                  'fallos': len(fallos), 'alucinación': clases['alucinación'],
                  '% alucinación': f"{clases['alucinación'] / len(fallos):.0%}"})
display(pd.DataFrame(filas).set_index('modelo'))

## 6. Guardar resultados

Colab borra todo al cerrar la sesión. Descarga `resultados/` (trazas, tablas y figuras) y `cache/` (las páginas nuevas de Wikipedia que se consultaron). Después se copian al repo local y se suben con git.

In [ ]:
!zip -qr salida.zip resultados cache
from google.colab import files
files.download('salida.zip')